# Notebook 03 - Pipeline Consolidado

**Tech Challenge - Fase 5 - Hackaton FIAP**
**Modelagem de Ameaças utilizando IA (STRIDE)**

Pipeline completo: Imagem -> Provedor -> Componentes -> STRIDE -> Relatório

Executa para as 2 arquiteturas de teste (AWS e Azure) e apresenta:
- Visualização com bounding boxes
- Resumo do relatório STRIDE
- Métricas de execução

**GPU necessária**: T4 ou A100

> **Configuração necessária antes de executar:**
> 1. No menu lateral do Colab, clique em 🔑 **Secrets** e adicione:
>    - `ANTHROPIC_API_KEY`: sua chave de API da Anthropic (obtenha em https://console.anthropic.com)
> 2. No menu **Ambiente de execução > Alterar tipo de ambiente de execução**, selecione:
>    - **Versão do ambiente de execução**: `2025.07`
>    - **GPU T4** (gratuita) ou **A100** (Colab Pro) — necessário para Florence-2

## 1. Setup

In [ ]:
!pip install -q transformers anthropic supervision timm einops
# Nota: flash_attn (Flash Attention) não é utilizada — a compilação falha no Colab
# por incompatibilidade com CUDA. Florence-2 funciona normalmente sem ela.

In [ ]:
import os
import io
import json
import base64
import time
from pathlib import Path
from datetime import datetime
from collections import Counter

import torch
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import supervision as sv
from transformers import AutoProcessor, AutoModelForCausalLM
import anthropic

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

DRIVE_BASE = Path('/content/drive/MyDrive/hackaton-stride')
IMAGES_DIR = DRIVE_BASE / 'test_images'
OUTPUT_DIR = DRIVE_BASE / 'outputs'
DETECTIONS_DIR = OUTPUT_DIR / 'detections'
REPORTS_DIR = OUTPUT_DIR / 'reports'

# Criar todos os diretórios necessários
for d in [IMAGES_DIR, DETECTIONS_DIR, REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY')
client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

## 2. Carregar Florence-2

In [ ]:
FLORENCE_MODEL_ID = 'microsoft/Florence-2-large-ft'

print("Carregando Florence-2-large-ft...")
florence_processor = AutoProcessor.from_pretrained(
    FLORENCE_MODEL_ID, trust_remote_code=True
)
florence_model = AutoModelForCausalLM.from_pretrained(
    FLORENCE_MODEL_ID, trust_remote_code=True,
    torch_dtype=torch.float16
).to('cuda')
print("Florence-2-large-ft pronto!")

## 3. Definir Pipeline Completo

In [ ]:
# Classes de detecção
COMPONENT_CLASSES = {
    'waf_firewall': ['AWS WAF', 'AWS Shield', 'Azure Firewall'],
    'cdn': ['Amazon CloudFront', 'Azure CDN'],
    'load_balancer': ['ALB', 'NLB', 'Azure Load Balancer'],
    'vpc_vnet': ['VPC', 'VNet', 'Subnet'],
    'compute': ['EC2', 'SEI/SIP', 'App Service', 'VM'],
    'auto_scaling': ['Auto Scaling Group', 'VMSS'],
    'orchestrator': ['Logic Apps', 'Step Functions', 'Lambda'],
    'database': ['RDS', 'Aurora', 'Azure SQL', 'Cosmos DB'],
    'cache': ['ElastiCache', 'Azure Cache for Redis'],
    'storage': ['S3', 'EFS', 'Azure Blob', 'NFS'],
    'api_gateway': ['API Gateway', 'Azure API Management'],
    'developer_portal': ['Developer Portal'],
    'web_service': ['REST', 'SOAP', 'SaaS Service'],
    'auth_identity': ['IAM', 'Microsoft Entra', 'Cognito'],
    'kms_encryption': ['AWS KMS', 'Azure Key Vault'],
    'monitoring': ['CloudWatch', 'CloudTrail', 'Azure Monitor'],
    'backup': ['AWS Backup', 'Azure Backup'],
    'messaging': ['SES', 'SQS', 'SNS', 'Azure Service Bus'],
    'user_actor': ['Usuário', 'Developer', 'Client'],
}

ALL_SERVICES = []
for cls, services in COMPONENT_CLASSES.items():
    for svc in services:
        ALL_SERVICES.append(f'{svc} ({cls})')
SERVICES_LIST = ', '.join(ALL_SERVICES)

In [ ]:
# === Funções auxiliares ===
import re

def encode_image_base64(image_path: str) -> str:
    with open(image_path, 'rb') as f:
        return base64.b64encode(f.read()).decode('utf-8')

def encode_pil_image_base64(pil_image: Image.Image) -> str:
    buffer = io.BytesIO()
    pil_image.save(buffer, format='PNG')
    return base64.b64encode(buffer.getvalue()).decode('utf-8')

def clean_json_text(text: str) -> str:
    """
    Limpa texto retornado pelo Claude para extrair JSON válido.
    Remove code fences, trailing commas e outros artefatos.
    """
    text = re.sub(r'```(?:json)?\s*\n?', '', text).strip()
    text = re.sub(r',\s*([}\]])', r'\1', text)
    return text


def parse_stride_json(text: str) -> dict:
    """
    Parser robusto para JSON STRIDE retornado pelo Claude.
    Tenta múltiplas estratégias de parsing.
    """
    # Estratégia 1: limpeza padrão + parse completo
    try:
        cleaned = clean_json_text(text)
        start = cleaned.find('{')
        end = cleaned.rfind('}') + 1
        if start >= 0 and end > start:
            return json.loads(cleaned[start:end])
    except json.JSONDecodeError:
        pass

    # Estratégia 2: extrair bloco bruto entre { }
    try:
        raw = text[text.find('{'):text.rfind('}') + 1]
        raw = re.sub(r',\s*([}\]])', r'\1', raw)
        return json.loads(raw)
    except (json.JSONDecodeError, ValueError):
        pass

    # Estratégia 3: extrair cada categoria STRIDE individualmente via regex
    categories = [
        'spoofing', 'tampering', 'repudiation',
        'information_disclosure', 'denial_of_service', 'elevation_of_privilege'
    ]
    result = {}
    for cat in categories:
        pattern = rf'"{cat}"\s*:\s*\{{([^}}]*)\}}'
        match = re.search(pattern, text, re.DOTALL)
        if match:
            try:
                inner = match.group(1)
                threat = re.search(r'"threat"\s*:\s*"((?:[^"\\]|\\.)*)"', inner)
                risk = re.search(r'"risk"\s*:\s*"((?:[^"\\]|\\.)*)"', inner)
                counter = re.search(r'"countermeasure"\s*:\s*"((?:[^"\\]|\\.)*)"', inner)
                ref = re.search(r'"reference"\s*:\s*"((?:[^"\\]|\\.)*)"', inner)
                result[cat] = {
                    'threat': threat.group(1) if threat else 'N/A',
                    'risk': risk.group(1) if risk else 'Médio',
                    'countermeasure': counter.group(1) if counter else 'N/A',
                    'reference': ref.group(1) if ref else 'N/A',
                }
            except Exception:
                pass

    return result if result else {}


def deduplicate_components(components: list) -> list:
    """
    Remove componentes duplicados após classificação Claude.
    Agrupa por nome e, para cada nome, mantém apenas instâncias
    espacialmente distintas (centros distantes > 100px).
    De cada cluster espacial, mantém o bbox de menor área (mais preciso).
    """
    from collections import defaultdict
    import math

    if not components:
        return components

    # Agrupar por nome
    by_name = defaultdict(list)
    for comp in components:
        by_name[comp['name']].append(comp)

    result = []
    for name, group in by_name.items():
        if len(group) == 1:
            result.append(group[0])
            continue

        # Clusterizar por proximidade espacial
        # bbox é [x, y, w, h] — calcular centro
        centers = []
        for c in group:
            cx = c['bbox'][0] + c['bbox'][2] / 2
            cy = c['bbox'][1] + c['bbox'][3] / 2
            area = c['bbox'][2] * c['bbox'][3]
            centers.append((cx, cy, area, c))

        clusters = []
        used = set()
        for i, (cx1, cy1, a1, c1) in enumerate(centers):
            if i in used:
                continue
            cluster = [(a1, c1)]
            used.add(i)
            for j, (cx2, cy2, a2, c2) in enumerate(centers):
                if j in used:
                    continue
                dist = math.sqrt((cx1 - cx2)**2 + (cy1 - cy2)**2)
                if dist < 100:  # mesmo componente se centros < 100px
                    cluster.append((a2, c2))
                    used.add(j)
            # Do cluster, manter o de menor área (mais preciso)
            cluster.sort(key=lambda x: x[0])
            result.append(cluster[0][1])

    # Reordenar por posição (top-left para bottom-right)
    result.sort(key=lambda c: (c['bbox'][1], c['bbox'][0]))

    # Re-numerar IDs
    for i, comp in enumerate(result):
        comp['id'] = i

    return result


# === Passo 1: Identificar provedor ===

def identify_provider(image_path: str) -> str:
    image_b64 = encode_image_base64(image_path)
    response = client.messages.create(
        model='claude-sonnet-4-6',
        max_tokens=100,
        messages=[{
            'role': 'user',
            'content': [
                {'type': 'image', 'source': {'type': 'base64', 'media_type': 'image/png', 'data': image_b64}},
                {'type': 'text', 'text': 'Analise esta imagem de diagrama de arquitetura de software. Identifique o provedor cloud principal. Responda APENAS: "AWS" ou "Azure".'}
            ]
        }]
    )
    text = response.content[0].text.strip().upper()
    return 'AWS' if 'AWS' in text else 'Azure' if 'AZURE' in text else 'AWS'


# === Passo 2a: Detecção com Florence-2 (múltiplas estratégias + tiling) ===

def run_florence_task(image: Image.Image, task: str, text_input: str = None) -> dict:
    prompt = task if text_input is None else task + text_input
    inputs = florence_processor(
        text=prompt, images=image, return_tensors='pt'
    ).to('cuda', torch.float16)
    with torch.no_grad():
        generated_ids = florence_model.generate(
            input_ids=inputs['input_ids'],
            pixel_values=inputs['pixel_values'],
            max_new_tokens=4096, num_beams=3
        )
    generated_text = florence_processor.batch_decode(
        generated_ids, skip_special_tokens=False
    )[0]
    return florence_processor.post_process_generation(
        generated_text, task=task,
        image_size=(image.width, image.height)
    )


def get_grounding_phrases(provider: str) -> list:
    phrases = [
        "icon", "logo", "symbol", "box", "label", "component", "service",
        "database", "server", "firewall", "load balancer", "storage", "gateway",
        "cloud", "network", "security", "monitoring", "backup", "cache",
        "user", "developer", "client", "actor",
        "arrow", "rectangle", "text", "diagram",
        "subnet", "zone", "region", "group",
    ]
    if provider == 'AWS':
        phrases.extend([
            "AWS WAF", "AWS Shield", "CloudFront", "ALB", "NLB",
            "VPC", "Subnet", "EC2", "Auto Scaling", "Auto Scaling Group",
            "RDS", "RDS Primary", "RDS Secondary", "Aurora",
            "ElastiCache", "EFS", "S3", "NFS",
            "CloudTrail", "CloudWatch", "KMS", "Key Management Service",
            "Backup", "AWS Backup",
            "SES", "Simple Email Service",
            "API Gateway", "Lambda", "Step Functions",
            "SQS", "SNS", "Cognito", "IAM",
            "Availability Zone", "Multi-AZ",
            "SEI", "SIP", "Solr",
            "Public Subnet", "Private Subnet",
            "Application Load Balancer", "Elastic File System",
        ])
    else:
        phrases.extend([
            "Azure Firewall", "Azure CDN", "Load Balancer",
            "VNet", "Subnet", "App Service", "VM", "VMSS",
            "Azure SQL", "Cosmos DB", "Azure Blob", "Key Vault",
            "Azure Monitor", "Azure Backup", "Service Bus",
            "API Management", "API Gateway", "Logic Apps",
            "Microsoft Entra", "Entra ID",
            "Developer Portal", "REST", "SOAP",
            "Resource Group", "HTTP", "HTTPS",
            "SaaS", "SaaS Service", "Web Service",
            "Azure Services", "Backend",
            "workflow", "authentication", "identity",
        ])
    return phrases


def run_all_detections(image: Image.Image, phrases: list, offset_x: int = 0, offset_y: int = 0) -> dict:
    boxes, labels = [], []
    def add_boxes(new_boxes, new_labels):
        for bbox in new_boxes:
            x1, y1, x2, y2 = bbox
            boxes.append([x1 + offset_x, y1 + offset_y, x2 + offset_x, y2 + offset_y])
        labels.extend(new_labels)

    try:
        result = run_florence_task(image, '<OCR_WITH_REGION>')
        if '<OCR_WITH_REGION>' in result:
            data = result['<OCR_WITH_REGION>']
            for qbox, label in zip(data.get('quad_boxes', []), data.get('labels', [])):
                if len(qbox) == 8:
                    x_coords = [qbox[i] for i in range(0, 8, 2)]
                    y_coords = [qbox[i] for i in range(1, 8, 2)]
                    boxes.append([min(x_coords)+offset_x, min(y_coords)+offset_y,
                                  max(x_coords)+offset_x, max(y_coords)+offset_y])
                    labels.append(label.strip())
    except Exception: pass

    for phrase in phrases:
        try:
            result = run_florence_task(image, '<CAPTION_TO_PHRASE_GROUNDING>', phrase)
            if '<CAPTION_TO_PHRASE_GROUNDING>' in result:
                data = result['<CAPTION_TO_PHRASE_GROUNDING>']
                add_boxes(data.get('bboxes', []), data.get('labels', []))
        except Exception: pass

    for task in ['<OD>', '<DENSE_REGION_CAPTION>', '<REGION_PROPOSAL>']:
        try:
            result = run_florence_task(image, task)
            if task in result:
                data = result[task]
                add_boxes(data.get('bboxes', []), data.get('labels', ['region'] * len(data.get('bboxes', []))))
        except Exception: pass

    return {'bboxes': boxes, 'labels': labels}


def detect_florence(image: Image.Image, provider: str) -> dict:
    phrases = get_grounding_phrases(provider)
    print(f"    [1/2] Imagem completa ({len(phrases)} phrases)...")
    full_result = run_all_detections(image, phrases)
    print(f"      -> {len(full_result['bboxes'])} regiões")
    all_boxes = list(full_result['bboxes'])
    all_labels = list(full_result['labels'])

    print("    [2/2] Tiling 3x3 com 20% overlap...")
    w, h = image.size
    cols, rows = 3, 3
    tile_w = int(w / cols * 1.2)
    tile_h = int(h / rows * 1.2)
    step_x = (w - tile_w) / (cols - 1) if cols > 1 else 0
    step_y = (h - tile_h) / (rows - 1) if rows > 1 else 0
    tile_count = 0
    for row in range(rows):
        for col in range(cols):
            tx1, ty1 = int(col * step_x), int(row * step_y)
            tx2, ty2 = min(tx1 + tile_w, w), min(ty1 + tile_h, h)
            tile = image.crop((tx1, ty1, tx2, ty2))
            tile_result = run_all_detections(tile, phrases, offset_x=tx1, offset_y=ty1)
            all_boxes.extend(tile_result['bboxes'])
            all_labels.extend(tile_result['labels'])
            tile_count += len(tile_result['bboxes'])
    print(f"      -> {tile_count} regiões por tiling")
    return {'bboxes': all_boxes, 'labels': all_labels}


def filter_boxes(detections: dict, image_size: tuple) -> dict:
    """
    Filtra bounding boxes e remove duplicatas via NMS.
    max_area_ratio=0.05 elimina detecções gigantes (>5% da imagem)
    que são artefatos de frases genéricas como 'box', 'icon'.
    """
    img_w, img_h = image_size
    img_area = img_w * img_h
    boxes, labels = [], []
    for bbox, label in zip(detections['bboxes'], detections['labels']):
        x1, y1, x2, y2 = bbox
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(img_w, x2), min(img_h, y2)
        w, h = x2 - x1, y2 - y1
        if 0.0005 <= (w*h)/img_area <= 0.05 and w > 5 and h > 5:
            boxes.append([x1, y1, x2, y2])
            labels.append(label)
    if not boxes: return {'bboxes': [], 'labels': []}
    boxes_np = np.array(boxes)
    keep, used = [], set()
    for i in range(len(boxes_np)):
        if i in used: continue
        keep.append(i)
        for j in range(i+1, len(boxes_np)):
            if j in used: continue
            xi1, yi1 = max(boxes_np[i][0], boxes_np[j][0]), max(boxes_np[i][1], boxes_np[j][1])
            xi2, yi2 = min(boxes_np[i][2], boxes_np[j][2]), min(boxes_np[i][3], boxes_np[j][3])
            inter = max(0, xi2-xi1) * max(0, yi2-yi1)
            a_i = (boxes_np[i][2]-boxes_np[i][0]) * (boxes_np[i][3]-boxes_np[i][1])
            a_j = (boxes_np[j][2]-boxes_np[j][0]) * (boxes_np[j][3]-boxes_np[j][1])
            if inter / (a_i + a_j - inter + 1e-6) > 0.5: used.add(j)
    return {'bboxes': [boxes[i] for i in keep], 'labels': [labels[i] for i in keep]}


# === Passo 2b: Classificação com Claude Vision ===

def classify_crop(crop: Image.Image, provider: str, full_b64: str) -> dict:
    crop_b64 = encode_pil_image_base64(crop)
    prompt = f"""Analise este recorte de um diagrama de arquitetura {provider}.
A segunda imagem mostra o diagrama completo para contexto.
Identifique o componente/serviço cloud.
Serviços possíveis: {SERVICES_LIST}
IMPORTANTE: Responda APENAS com JSON puro, sem code fences, sem comentários, sem texto antes ou depois.
{{"name": "Nome do serviço", "class": "classe"}}
Se não identificável: {{"name": "unknown", "class": "unknown"}}"""
    response = client.messages.create(
        model='claude-sonnet-4-6', max_tokens=200,
        messages=[{'role': 'user', 'content': [
            {'type': 'image', 'source': {'type': 'base64', 'media_type': 'image/png', 'data': crop_b64}},
            {'type': 'image', 'source': {'type': 'base64', 'media_type': 'image/png', 'data': full_b64}},
            {'type': 'text', 'text': prompt}
        ]}]
    )
    text = response.content[0].text.strip()
    try:
        cleaned = clean_json_text(text)
        s, e = cleaned.find('{'), cleaned.rfind('}') + 1
        if s >= 0 and e > s: return json.loads(cleaned[s:e])
    except json.JSONDecodeError: pass
    return {'name': 'unknown', 'class': 'unknown'}


# === Passo 2c: Análise de Topologia com Claude Vision ===

def analyze_topology(image_path: str, components: list, provider: str) -> dict:
    image_b64 = encode_image_base64(image_path)
    comp_list = "\n".join(f"  - {c['name']} ({c['class']})" for c in components)
    prompt = f"""Analise este diagrama de arquitetura {provider}.

Os seguintes componentes foram detectados:
{comp_list}

Identifique a topologia do diagrama:
1. Grupos/containers (VPC, Subnets, AZs, Resource Groups)
2. Conexões entre componentes (setas/linhas)
3. Fluxo de dados de ponta a ponta

IMPORTANTE: Responda APENAS com JSON puro, sem code fences, sem comentários.
{{
  "groups": [{{"name": "...", "type": "...", "contains": ["..."], "parent": "... ou null"}}],
  "connections": [{{"from": "...", "to": "...", "description": "..."}}],
  "data_flow": ["Componente 1", "Componente 2", "..."]
}}

Use EXATAMENTE os nomes dos componentes listados acima."""
    response = client.messages.create(
        model='claude-sonnet-4-6', max_tokens=4000,
        messages=[{'role': 'user', 'content': [
            {'type': 'image', 'source': {'type': 'base64', 'media_type': 'image/png', 'data': image_b64}},
            {'type': 'text', 'text': prompt}
        ]}]
    )
    text = response.content[0].text.strip()
    try:
        cleaned = clean_json_text(text)
        s, e = cleaned.find('{'), cleaned.rfind('}') + 1
        if s >= 0 and e > s: return json.loads(cleaned[s:e])
    except json.JSONDecodeError: pass
    return {'groups': [], 'connections': [], 'data_flow': []}


# === Passo 3: Análise STRIDE (com topologia) ===

def get_component_topology_context(component: dict, topology: dict) -> str:
    lines = []
    comp_name = component['name']
    for g in topology.get('groups', []):
        if comp_name in g.get('contains', []):
            parent = f" (dentro de {g['parent']})" if g.get('parent') else ""
            lines.append(f"  - Está no grupo: {g['name']} [{g.get('type', '?')}]{parent}")
    for conn in topology.get('connections', []):
        if conn['from'] == comp_name:
            lines.append(f"  - Conecta para: {conn['to']} ({conn.get('description', '')})")
        elif conn['to'] == comp_name:
            lines.append(f"  - Recebe de: {conn['from']} ({conn.get('description', '')})")
    data_flow = topology.get('data_flow', [])
    if comp_name in data_flow:
        lines.append(f"  - Posição no fluxo: {data_flow.index(comp_name) + 1}/{len(data_flow)}")
    return "\n".join(lines) if lines else "  (sem informação topológica)"


def analyze_stride(component: dict, all_components: list, provider: str,
                   topology: dict) -> dict:
    arch_context = "\n".join(f"  - {c['name']} ({c['class']})" for c in all_components)
    topo_context = get_component_topology_context(component, topology)
    data_flow = topology.get('data_flow', [])
    flow_str = " -> ".join(data_flow) if data_flow else "(não disponível)"

    prompt = f"""Você é um especialista em segurança de software.

Analise o componente usando STRIDE. Arquitetura {provider}.

Componente: {component['name']} ({component['class']})

Posição na topologia:
{topo_context}

Fluxo de dados: {flow_str}

Outros componentes:
{arch_context}

Para cada categoria STRIDE, forneça: threat, risk (Alto/Médio/Baixo), countermeasure, reference.
Considere a posição topológica e conexões do componente.
Contramedidas específicas de {provider}. Referencie {"AWS Well-Architected" if provider == "AWS" else "Azure Security Benchmark"}.
Máximo 150 caracteres por campo de texto. Não use aspas duplas dentro dos valores.

IMPORTANTE: Responda APENAS com JSON puro, sem code fences (```), sem comentários, sem texto antes ou depois.
{{"spoofing": {{"threat": "...", "risk": "Alto|Médio|Baixo", "countermeasure": "...", "reference": "..."}},
"tampering": {{...}}, "repudiation": {{...}}, "information_disclosure": {{...}},
"denial_of_service": {{...}}, "elevation_of_privilege": {{...}}}}"""

    response = client.messages.create(
        model='claude-sonnet-4-6', max_tokens=2000,
        messages=[{'role': 'user', 'content': prompt}]
    )
    text = response.content[0].text.strip()
    result = parse_stride_json(text)

    if not result:
        print(f"  Erro: não foi possível parsear JSON para {component['name']}")

    return result

In [ ]:
# === Passo 4: Geração de relatório HTML ===

def generate_summary(stride_results: list) -> dict:
    total_threats = 0
    risk_counts = {'Alto': 0, 'Médio': 0, 'Baixo': 0}
    high_risk = []
    for r in stride_results:
        h = 0
        for cat, d in r.get('threats', {}).items():
            if isinstance(d, dict):
                total_threats += 1
                risk = d.get('risk', 'Baixo')
                if risk in risk_counts: risk_counts[risk] += 1
                if risk == 'Alto': h += 1
        if h > 0:
            high_risk.append({'name': r['name'], 'class': r['class'], 'high_risk_count': h})
    return {
        'total_components': len(stride_results),
        'total_threats': total_threats,
        'risk_distribution': risk_counts,
        'high_risk_components': sorted(high_risk, key=lambda x: x['high_risk_count'], reverse=True)
    }


def generate_html_report(stride_results: list, summary: dict, provider: str,
                         arch_index: int, topology: dict = None,
                         original_image_b64: str = None,
                         annotated_image_b64: str = None) -> str:
    """Gera relatório HTML completo da análise STRIDE com topologia e imagens."""

    category_names = {
        'spoofing': 'S - Spoofing', 'tampering': 'T - Tampering',
        'repudiation': 'R - Repudiation', 'information_disclosure': 'I - Info Disclosure',
        'denial_of_service': 'D - Denial of Service', 'elevation_of_privilege': 'E - Elevation'
    }
    risk_colors = {'Alto': ('#dc3545', '#fff'), 'Médio': ('#ffc107', '#333'), 'Baixo': ('#28a745', '#fff')}
    generated_at = datetime.now().strftime('%d/%m/%Y %H:%M')
    if topology is None:
        topology = {'groups': [], 'connections': [], 'data_flow': []}

    # Mapa de numeração
    comp_number = {r['name']: i + 1 for i, r in enumerate(stride_results)}

    # Mapa de localização topológica
    def get_component_location(comp_name):
        locations = []
        for g in topology.get('groups', []):
            if comp_name in g.get('contains', []):
                locations.append(g['name'])
        return locations[-1] if locations else ''

    comp_location = {r['name']: get_component_location(r['name']) for r in stride_results}

    # Detectar nomes duplicados
    name_counts = Counter(r['name'] for r in stride_results)
    has_duplicates = {name for name, count in name_counts.items() if count > 1}

    # --- Imagens ---
    images_html = ''
    if original_image_b64 or annotated_image_b64:
        images_html = '<h1 style="margin-top:40px">Diagrama da Arquitetura</h1>'
        if original_image_b64:
            images_html += '<h3>Imagem Original</h3><div style="text-align:center;margin-bottom:20px"><img src="data:image/png;base64,' + original_image_b64 + '" style="max-width:100%;border:1px solid #ddd;border-radius:8px;box-shadow:0 2px 8px rgba(0,0,0,0.1)" alt="Diagrama original"></div>'
        if annotated_image_b64:
            images_html += '<h3>Componentes Detectados</h3><div style="text-align:center;margin-bottom:20px"><img src="data:image/png;base64,' + annotated_image_b64 + '" style="max-width:100%;border:1px solid #ddd;border-radius:8px;box-shadow:0 2px 8px rgba(0,0,0,0.1)" alt="Componentes detectados"></div>'

    # --- Tabela de componentes (com localização) ---
    comp_rows = ''
    for i, r in enumerate(stride_results):
        bg = '#f9f9f9' if i % 2 == 0 else '#fff'
        loc = comp_location.get(r['name'], '')
        loc_html = f'<span style="color:#666;font-size:12px">{loc}</span>' if loc else '<span style="color:#ccc;font-size:12px">—</span>'
        comp_rows += f'<tr style="background:{bg}"><td style="padding:6px 10px;border:1px solid #ddd;text-align:center">{i+1}</td><td style="padding:6px 10px;border:1px solid #ddd">{r["name"]}</td><td style="padding:6px 10px;border:1px solid #ddd;text-align:center">{r["class"]}</td><td style="padding:6px 10px;border:1px solid #ddd">{loc_html}</td><td style="padding:6px 10px;border:1px solid #ddd;text-align:center">{r["provider"]}</td></tr>'

    # High risk (com numeração e localização)
    high_risk_html = ''
    if summary.get('high_risk_components'):
        items = ''
        for c in summary['high_risk_components']:
            num = comp_number.get(c['name'], '?')
            loc = comp_location.get(c['name'], '')
            loc_text = f' <span style="color:#999;font-size:12px">({loc})</span>' if loc else ''
            items += f"<li><strong>#{num} {c['name']}</strong>{loc_text}: {c['high_risk_count']} riscos altos</li>"
        high_risk_html = f'<h3 style="color:#dc3545;margin-top:20px">Componentes Críticos</h3><ul>{items}</ul>'

    # --- Topologia ---
    topology_html = ''
    data_flow = topology.get('data_flow', [])
    groups = topology.get('groups', [])
    connections = topology.get('connections', [])

    if data_flow or groups or connections:
        topology_html = '<h1 style="margin-top:40px">Topologia da Arquitetura</h1>'
        if data_flow:
            flow_steps = ''
            for i, step in enumerate(data_flow):
                num = comp_number.get(step, '?')
                flow_steps += f'<span style="display:inline-block;background:#e3f2fd;border:1px solid #90caf9;padding:4px 12px;border-radius:6px;font-size:13px;font-weight:bold">#{num} {step}</span>'
                if i < len(data_flow) - 1:
                    flow_steps += '<span style="display:inline-block;padding:0 6px;color:#90caf9;font-size:18px;vertical-align:middle">&#10140;</span>'
            topology_html += f'<h3>Fluxo de Dados</h3><div style="padding:15px;background:#fafafa;border:1px solid #e0e0e0;border-radius:8px;line-height:2.2;margin-bottom:20px">{flow_steps}</div>'
        if groups:
            groups_html = ''
            for g in groups:
                parent_info = f' <span style="color:#999;font-size:12px">(dentro de {g["parent"]})</span>' if g.get('parent') else ''
                contains = g.get('contains', [])
                chips = ' '.join(f'<span style="display:inline-block;background:#fff;border:1px solid #ccc;padding:2px 8px;border-radius:4px;font-size:12px;margin:2px">#{comp_number.get(c, "?")} {c}</span>' for c in contains) if contains else '<span style="color:#999;font-size:12px">vazio</span>'
                groups_html += f'<div style="margin-bottom:10px;padding:10px 14px;background:#f5f5f5;border-left:4px solid #1565c0;border-radius:4px"><strong>{g["name"]}</strong> <span style="color:#666;font-size:12px">[{g.get("type", "?")}]</span>{parent_info}<div style="margin-top:6px">{chips}</div></div>'
            topology_html += f'<h3>Grupos e Containers</h3>{groups_html}'
        if connections:
            conn_rows = ''
            for c in connections:
                from_num = comp_number.get(c['from'], '?')
                to_num = comp_number.get(c['to'], '?')
                conn_rows += f'<tr><td style="padding:5px 10px;border:1px solid #ddd">#{from_num} {c["from"]}</td><td style="padding:5px 10px;border:1px solid #ddd;text-align:center">&#10140;</td><td style="padding:5px 10px;border:1px solid #ddd">#{to_num} {c["to"]}</td><td style="padding:5px 10px;border:1px solid #ddd;color:#666;font-size:12px">{c.get("description", "")}</td></tr>'
            topology_html += f'<h3 style="margin-top:20px">Conexões</h3><table><thead><tr><th>Origem</th><th style="width:30px"></th><th>Destino</th><th>Descrição</th></tr></thead><tbody>{conn_rows}</tbody></table>'

    # Nota sobre componentes repetidos
    duplicate_note = ''
    if has_duplicates:
        duplicate_note = '<div style="margin:20px 0;padding:12px 16px;background:#fff3cd;border:1px solid #ffc107;border-radius:6px;font-size:13px;color:#856404"><strong>Nota:</strong> Componentes com o mesmo nome aparecem mais de uma vez quando estão em diferentes zonas de disponibilidade ou grupos. A análise STRIDE de cada instância considera sua posição específica na topologia, conexões e fluxo de dados, podendo resultar em riscos e contramedidas distintas.</div>'

    # STRIDE details (com localização e conexões)
    stride_details = ''
    for i, result in enumerate(stride_results):
        num = i + 1
        threats = result.get('threats', {})
        loc = comp_location.get(result['name'], '')
        loc_badge = f' <span style="background:#e3f2fd;color:#1565c0;padding:2px 10px;border-radius:10px;font-size:12px;font-weight:normal">{loc}</span>' if loc else ''

        # Conexões deste componente
        comp_conns = []
        for conn in topology.get('connections', []):
            if conn.get('from') == result['name']:
                comp_conns.append(f"→ {conn['to']}")
            elif conn.get('to') == result['name']:
                comp_conns.append(f"← {conn['from']}")
        conns_html = ''
        if comp_conns:
            conns_chips = ' '.join(f'<span style="display:inline-block;background:#f5f5f5;border:1px solid #ddd;padding:1px 8px;border-radius:4px;font-size:11px;margin:1px">{c}</span>' for c in comp_conns[:6])
            if len(comp_conns) > 6:
                conns_chips += f' <span style="color:#999;font-size:11px">+{len(comp_conns)-6} mais</span>'
            conns_html = f'<div style="margin-top:4px;margin-bottom:10px"><strong style="font-size:12px;color:#666">Conexões:</strong> {conns_chips}</div>'

        cats_html = ''
        for cat_key, cat_name in category_names.items():
            if cat_key not in threats or not isinstance(threats[cat_key], dict): continue
            d = threats[cat_key]
            risk = d.get('risk', 'Baixo')
            bg_color, fg_color = risk_colors.get(risk, ('#6c757d', '#fff'))
            cats_html += f'''<div style="margin-bottom:12px;padding:10px;background:#f8f9fa;border-left:4px solid {bg_color};border-radius:4px">
                <div style="display:flex;justify-content:space-between;align-items:center;margin-bottom:6px">
                    <strong style="font-size:14px">{cat_name}</strong>
                    <span style="background:{bg_color};color:{fg_color};padding:2px 10px;border-radius:12px;font-size:12px;font-weight:bold">{risk}</span>
                </div>
                <div style="font-size:13px;line-height:1.5">
                    <div><strong>Ameaça:</strong> {d.get('threat', 'N/A')}</div>
                    <div><strong>Contramedida:</strong> {d.get('countermeasure', 'N/A')}</div>
                    <div style="color:#666;font-size:12px"><strong>Ref:</strong> {d.get('reference', 'N/A')}</div>
                </div></div>'''
        stride_details += f'''<div style="page-break-before:always;margin-top:30px">
            <h2 style="color:#333;border-bottom:2px solid #007bff;padding-bottom:5px">#{num} {result['name']}{loc_badge}</h2>
            <p style="color:#666;margin-bottom:5px">Classe: {result['class']} | Provedor: {result['provider']}</p>
            {conns_html}
            {cats_html}</div>'''

    # Risk badges
    risk_badges = ''
    for level in ['Alto', 'Médio', 'Baixo']:
        count = summary['risk_distribution'].get(level, 0)
        bg_color, fg_color = risk_colors.get(level, ('#6c757d', '#fff'))
        risk_badges += f'<span style="display:inline-block;background:{bg_color};color:{fg_color};padding:4px 14px;border-radius:14px;margin-right:10px;font-weight:bold">{level}: {count}</span>'

    return f'''<!DOCTYPE html>
<html lang="pt-BR">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Relatório STRIDE - Arquitetura {arch_index} ({provider})</title>
    <style>
        body {{ font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Arial, sans-serif; margin:0; padding:0; color:#333; line-height:1.6; }}
        .cover {{ text-align:center; padding:50px 20px; background:linear-gradient(135deg, #2196F3 0%, #1976D2 50%, #1565C0 100%); color:white; }}
        .cover h1, .cover h2, .cover p {{ color:white; }}
        .cover h1 {{ font-size:36px; margin-bottom:5px; }}
        .cover h2 {{ font-size:24px; font-weight:normal; opacity:0.9; }}
        .cover p {{ font-size:14px; opacity:0.8; margin-top:30px; }}
        .content {{ max-width:900px; margin:0 auto; padding:30px 20px; }}
        h1 {{ color:#1565C0; }}
        table {{ width:100%; border-collapse:collapse; margin:15px 0; }}
        th {{ background:#1976D2; color:white; padding:8px 10px; text-align:center; border:1px solid #ddd; }}
        @media print {{ .cover {{ page-break-after:always; }} div[style*="page-break-before"] {{ page-break-before:always; }} }}
    </style>
</head>
<body>
    <div class="cover">
        <h1>Relatório STRIDE</h1>
        <h2>Arquitetura {arch_index} - {provider}</h2>
        <p>Tech Challenge - Fase 5 - Hackaton FIAP<br>Gerado em: {generated_at}</p>
    </div>
    <div class="content">
        <h1>Resumo Executivo</h1>
        <p>Componentes: <strong>{summary['total_components']}</strong> | Ameaças: <strong>{summary['total_threats']}</strong></p>
        <p>{risk_badges}</p>
        {high_risk_html}
        {images_html}
        {topology_html}
        <h1 style="margin-top:40px">Componentes Detectados</h1>
        <table><thead><tr><th style="width:40px">#</th><th>Componente</th><th>Classe</th><th>Localização</th><th>Provedor</th></tr></thead><tbody>{comp_rows}</tbody></table>
        {duplicate_note}
        <h1 style="margin-top:40px">Análise STRIDE Detalhada</h1>
        {stride_details}
    </div>
</body>
</html>'''

## 4. Pipeline End-to-End

In [ ]:
def run_pipeline(arch_index: int) -> dict:
    """
    Executa o pipeline completo para uma arquitetura.
    Retorna dict com resultados e métricas.
    """
    print(f"{'='*60}")
    print(f"PIPELINE - Arquitetura {arch_index}")
    print(f"{'='*60}")
    metrics = {}
    t_total = time.time()

    # Carregar imagem
    image_path = IMAGES_DIR / f'arquitetura {arch_index}.png'
    image = Image.open(image_path).convert('RGB')
    print(f"\nImagem: {image_path.name} ({image.size})")

    # --- Passo 1: Identificar provedor ---
    print(f"\n[Passo 1] Identificando provedor cloud...")
    t0 = time.time()
    provider = identify_provider(str(image_path))
    metrics['step1_provider_s'] = round(time.time() - t0, 2)
    print(f"  Provedor: {provider} ({metrics['step1_provider_s']}s)")

    # --- Passo 2a: Detecção Florence-2 ---
    print(f"\n[Passo 2a] Detectando componentes com Florence-2 (múltiplas estratégias + tiling)...")
    t0 = time.time()
    raw = detect_florence(image, provider)
    detections = filter_boxes(raw, image.size)
    metrics['step2a_florence_s'] = round(time.time() - t0, 2)
    print(f"  Regiões após filtro + NMS: {len(detections['bboxes'])} ({metrics['step2a_florence_s']}s)")

    # --- Passo 2b: Classificação Claude Vision ---
    print(f"\n[Passo 2b] Classificando componentes com Claude Vision...")
    t0 = time.time()
    full_b64 = encode_image_base64(str(image_path))
    components = []
    for idx, bbox in enumerate(detections['bboxes']):
        x1, y1, x2, y2 = [int(c) for c in bbox]
        margin = 10
        crop = image.crop((
            max(0, x1-margin), max(0, y1-margin),
            min(image.width, x2+margin), min(image.height, y2+margin)
        ))
        result = classify_crop(crop, provider, full_b64)
        if result.get('class') != 'unknown':
            components.append({
                'id': idx, 'name': result['name'], 'class': result['class'],
                'bbox': [x1, y1, x2-x1, y2-y1], 'provider': provider
            })
            print(f"  [{idx+1}] {result['name']} ({result['class']})")
        time.sleep(0.5)
    metrics['step2b_classify_s'] = round(time.time() - t0, 2)
    print(f"  Componentes identificados: {len(components)} ({metrics['step2b_classify_s']}s)")

    # Deduplicar: agrupar por nome + proximidade espacial
    before_dedup = len(components)
    components = deduplicate_components(components)
    print(f"  Após deduplicação espacial: {len(components)} (removidos {before_dedup - len(components)} duplicatas)")
    metrics['components_detected'] = len(components)

    # --- Passo 2c: Análise de Topologia ---
    print(f"\n[Passo 2c] Analisando topologia com Claude Vision...")
    t0 = time.time()
    topology = analyze_topology(str(image_path), components, provider)
    metrics['step2c_topology_s'] = round(time.time() - t0, 2)
    n_groups = len(topology.get('groups', []))
    n_conns = len(topology.get('connections', []))
    n_flow = len(topology.get('data_flow', []))
    print(f"  Grupos: {n_groups} | Conexões: {n_conns} | Fluxo: {n_flow} passos ({metrics['step2c_topology_s']}s)")

    # Salvar detecções (com topologia)
    det_data = {
        'image': image_path.name, 'provider': provider,
        'image_size': {'width': image.width, 'height': image.height},
        'components': components,
        'topology': topology,
        'timing': {
            'step1_provider_s': metrics['step1_provider_s'],
            'step2a_florence_s': metrics['step2a_florence_s'],
            'step2b_classify_s': metrics['step2b_classify_s'],
            'step2c_topology_s': metrics['step2c_topology_s'],
        }
    }
    det_path = DETECTIONS_DIR / f'componentes_arquitetura_{arch_index}.json'
    with open(det_path, 'w', encoding='utf-8') as f:
        json.dump(det_data, f, ensure_ascii=False, indent=2)

    # --- Passo 3: Análise STRIDE (com topologia) ---
    print(f"\n[Passo 3] Executando análise STRIDE (com contexto topológico)...")
    t0 = time.time()
    stride_results = []
    for comp in components:
        threats = analyze_stride(comp, components, provider, topology)
        stride_results.append({**comp, 'threats': threats})
        risks = [threats[c].get('risk', '?') for c in threats if isinstance(threats[c], dict)]
        high = risks.count('Alto')
        print(f"  {comp['name']}: {high} riscos altos")
        time.sleep(1)
    metrics['step3_stride_s'] = round(time.time() - t0, 2)
    print(f"  Análise concluída ({metrics['step3_stride_s']}s)")

    # --- Passo 4: Relatórios ---
    print(f"\n[Passo 4] Gerando relatórios...")
    t0 = time.time()
    summary = generate_summary(stride_results)

    # JSON
    report = {
        'metadata': {
            'project': 'Tech Challenge - Fase 5 - Hackaton FIAP',
            'methodology': 'STRIDE',
            'generated_at': datetime.now().isoformat(),
            'image': image_path.name, 'provider': provider
        },
        'topology': topology,
        'components': stride_results, 'summary': summary
    }
    json_path = REPORTS_DIR / f'stride_arquitetura_{arch_index}.json'
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(report, f, ensure_ascii=False, indent=2)

    # Codificar imagens para o HTML (base64 inline)
    original_b64 = encode_image_base64(str(image_path))

    annotated_b64 = None
    annotated = None
    if components:
        xyxy = np.array([[c['bbox'][0], c['bbox'][1], c['bbox'][0]+c['bbox'][2], c['bbox'][1]+c['bbox'][3]] for c in components])
        det_sv = sv.Detections(xyxy=xyxy, class_id=np.arange(len(components)))
        labels = [f"#{i+1} {c['name']}" for i, c in enumerate(components)]
        img_np = np.array(image)
        annotated = sv.BoxAnnotator(thickness=2).annotate(scene=img_np.copy(), detections=det_sv)
        annotated = sv.LabelAnnotator(text_scale=0.4, text_padding=4).annotate(scene=annotated, detections=det_sv, labels=labels)
        ann_path = DETECTIONS_DIR / f'annotated_arquitetura_{arch_index}.png'
        Image.fromarray(annotated).save(ann_path)
        # Codificar imagem anotada em base64
        with open(ann_path, 'rb') as f:
            annotated_b64 = base64.b64encode(f.read()).decode('utf-8')

    # HTML (com topologia e imagens embutidas)
    html_content = generate_html_report(
        stride_results, summary, provider, arch_index, topology,
        original_image_b64=original_b64,
        annotated_image_b64=annotated_b64
    )
    html_path = REPORTS_DIR / f'stride_arquitetura_{arch_index}.html'
    with open(html_path, 'w', encoding='utf-8') as f:
        f.write(html_content)

    metrics['step4_reports_s'] = round(time.time() - t0, 2)
    metrics['total_threats'] = summary['total_threats']
    metrics['risk_distribution'] = summary['risk_distribution']
    metrics['topology'] = {'groups': n_groups, 'connections': n_conns, 'data_flow': n_flow}
    print(f"  JSON: {json_path}")
    print(f"  HTML: {html_path}")

    metrics['total_s'] = round(time.time() - t_total, 2)

    return {
        'arch_index': arch_index, 'provider': provider,
        'components': components, 'stride_results': stride_results,
        'summary': summary, 'metrics': metrics, 'topology': topology,
        'annotated': annotated
    }

## 5. Executar para Ambas as Arquiteturas

In [ ]:
# Executar pipeline para Arquitetura 1 (AWS)
result_1 = run_pipeline(1)

In [ ]:
# Executar pipeline para Arquitetura 2 (Azure)
result_2 = run_pipeline(2)

## 6. Visualização Comparativa

In [ ]:
# Visualização lado a lado: imagens com bounding boxes
fig, axes = plt.subplots(1, 2, figsize=(24, 12))

for ax, result in zip(axes, [result_1, result_2]):
    if result['annotated'] is not None:
        ax.imshow(result['annotated'])
    ax.set_title(
        f"Arquitetura {result['arch_index']} ({result['provider']})\n"
        f"{result['metrics']['components_detected']} componentes | "
        f"{result['metrics']['total_threats']} ameaças",
        fontsize=14
    )
    ax.axis('off')

plt.suptitle('Detecção de Componentes - Comparativo', fontsize=18, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Comparativo de riscos
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, result in zip(axes, [result_1, result_2]):
    risks = result['metrics']['risk_distribution']
    colors_bar = ['#dc3545', '#ffc107', '#28a745']
    bars = ax.bar(risks.keys(), risks.values(), color=colors_bar)
    ax.set_title(f"Arq. {result['arch_index']} ({result['provider']})")
    ax.set_ylabel('Quantidade de ameaças')
    for bar, val in zip(bars, risks.values()):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                str(val), ha='center', fontweight='bold')

plt.suptitle('Distribuição de Riscos STRIDE', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Métricas de Execução

In [ ]:
# Tabela comparativa de métricas
print("=" * 70)
print(f"{'MÉTRICAS DE EXECUÇÃO':^70}")
print("=" * 70)
print(f"{'Métrica':<35} {'Arq. 1 (AWS)':>15} {'Arq. 2 (Azure)':>15}")
print("-" * 70)

m1 = result_1['metrics']
m2 = result_2['metrics']

rows = [
    ('Passo 1 - Provedor (s)', m1['step1_provider_s'], m2['step1_provider_s']),
    ('Passo 2a - Florence-2 (s)', m1['step2a_florence_s'], m2['step2a_florence_s']),
    ('Passo 2b - Classificação (s)', m1['step2b_classify_s'], m2['step2b_classify_s']),
    ('Passo 2c - Topologia (s)', m1['step2c_topology_s'], m2['step2c_topology_s']),
    ('Passo 3 - STRIDE (s)', m1['step3_stride_s'], m2['step3_stride_s']),
    ('Passo 4 - Relatórios (s)', m1['step4_reports_s'], m2['step4_reports_s']),
    ('Tempo total (s)', m1['total_s'], m2['total_s']),
    ('', '', ''),
    ('Componentes detectados', m1['components_detected'], m2['components_detected']),
    ('Grupos topológicos', m1['topology']['groups'], m2['topology']['groups']),
    ('Conexões detectadas', m1['topology']['connections'], m2['topology']['connections']),
    ('Passos no fluxo', m1['topology']['data_flow'], m2['topology']['data_flow']),
    ('Total de ameaças', m1['total_threats'], m2['total_threats']),
    ('Riscos altos', m1['risk_distribution']['Alto'], m2['risk_distribution']['Alto']),
    ('Riscos médios', m1['risk_distribution']['Médio'], m2['risk_distribution']['Médio']),
    ('Riscos baixos', m1['risk_distribution']['Baixo'], m2['risk_distribution']['Baixo']),
]

for label, v1, v2 in rows:
    print(f"{label:<35} {str(v1):>15} {str(v2):>15}")

print("=" * 70)
print(f"\nArquivos gerados:")
for idx in [1, 2]:
    print(f"  Arq. {idx}:")
    print(f"    - {DETECTIONS_DIR}/componentes_arquitetura_{idx}.json")
    print(f"    - {DETECTIONS_DIR}/annotated_arquitetura_{idx}.png")
    print(f"    - {REPORTS_DIR}/stride_arquitetura_{idx}.json")
    print(f"    - {REPORTS_DIR}/stride_arquitetura_{idx}.html")

## 8. Estimativa de Custo

Estimativa do custo total do pipeline para as duas arquiteturas, considerando:
- **Claude API** (Sonnet 4.6): US$ 3,00/1M tokens input | US$ 15,00/1M tokens output
- **Google Colab Pro**: ~US$ 10/mês (~US$ 0,014/hora GPU T4)

In [ ]:
# Estimativa de custo total (2 arquiteturas)
PRICE_INPUT = 3.00 / 1_000_000   # US$/token input
PRICE_OUTPUT = 15.00 / 1_000_000  # US$/token output
COLAB_HOUR = 0.014                # US$/hora GPU T4
USD_TO_BRL = 5.50                 # taxa aproximada

print("=" * 70)
print(f"{'ESTIMATIVA DE CUSTO TOTAL':^70}")
print("=" * 70)

total_usd = 0

for result in [result_1, result_2]:
    n = result['metrics']['components_detected']
    arch = result['arch_index']
    provider = result['provider']

    # Passo 1: 1 chamada com imagem (~1.5k tokens input, ~10 output)
    p1_in, p1_out = 1500, 10

    # Passo 2b: N crops × (~2k input com 2 imagens, ~50 output)
    p2b_in, p2b_out = n * 2000, n * 50

    # Passo 2c: 1 chamada com imagem + lista (~2k input, ~1k output)
    p2c_in, p2c_out = 2000, 1000

    # Passo 3: N componentes × (~3.5k input, ~500 output)
    p3_in, p3_out = n * 3500, n * 500

    total_in = p1_in + p2b_in + p2c_in + p3_in
    total_out = p1_out + p2b_out + p2c_out + p3_out
    cost_usd = total_in * PRICE_INPUT + total_out * PRICE_OUTPUT

    print(f"\nArquitetura {arch} ({provider}) - {n} componentes:")
    print(f"  Passo 1 (Provedor):     ~{p1_in:>6} in + {p1_out:>6} out = US$ {(p1_in*PRICE_INPUT + p1_out*PRICE_OUTPUT):.4f}")
    print(f"  Passo 2b (Classific.):  ~{p2b_in:>6} in + {p2b_out:>6} out = US$ {(p2b_in*PRICE_INPUT + p2b_out*PRICE_OUTPUT):.4f}")
    print(f"  Passo 2c (Topologia):   ~{p2c_in:>6} in + {p2c_out:>6} out = US$ {(p2c_in*PRICE_INPUT + p2c_out*PRICE_OUTPUT):.4f}")
    print(f"  Passo 3 (STRIDE):       ~{p3_in:>6} in + {p3_out:>6} out = US$ {(p3_in*PRICE_INPUT + p3_out*PRICE_OUTPUT):.4f}")
    print(f"  Passo 4 (HTML):         gerado localmente (sem custo API)")
    print(f"  Subtotal Claude API:    US$ {cost_usd:.4f} (R$ {cost_usd * USD_TO_BRL:.4f})")
    total_usd += cost_usd

# Colab GPU
total_time_s = result_1['metrics']['total_s'] + result_2['metrics']['total_s']
colab_hours = total_time_s / 3600
colab_usd = colab_hours * COLAB_HOUR

print(f"\n{'─' * 70}")
print(f"Google Colab GPU T4:      {total_time_s:.0f}s = {colab_hours:.3f}h = US$ {colab_usd:.4f} (R$ {colab_usd * USD_TO_BRL:.4f})")
print(f"\n{'=' * 70}")
grand_total = total_usd + colab_usd
print(f"CUSTO TOTAL ESTIMADO:     US$ {grand_total:.4f} (R$ {grand_total * USD_TO_BRL:.4f})")
print(f"Custo médio por imagem:   US$ {grand_total/2:.4f} (R$ {grand_total/2 * USD_TO_BRL:.4f})")
print(f"{'=' * 70}")